# 06 – Ethics, Fairness & Bias
Audit hồi cứu của hai winner đã khóa, không phải đợt tối ưu mới. Công bố theo phân hoạch UK và Non-UK / Unknown; nhãn gộp không đồng nghĩa quốc tế. Không suy ra quốc tịch, giới tính, tuổi hoặc các nhóm bảo vệ từ Country. Không kết luận công bằng chỉ vì gap nhỏ. Không commit bảng cấp khách.

In [1]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src'/'fairness_evaluation.py').is_file()), None)
if ROOT is None: raise FileNotFoundError('Mở notebook trong repository.')
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src.pipeline import load_raw_data, clean_transactions, integrate_external_sources
from src.data_quality import missing_id_comparison
from src import classification_evaluation as ce, regression_evaluation as re
from src.fairness_evaluation import (market_groups, classification_by_group, regression_by_group,
    country_selection, privacy_summary, signed_gap, public_representation, update_cards)
from src.fairness_reference import reference_issues, classification_reference, regression_reference, publication_issues
REPORTS = ROOT/'reports'; REPORTS.mkdir(exist_ok=True)
raw = load_raw_data(ROOT/'data/raw')
observed_until = raw.InvoiceDate.max()
summary, _, country = missing_id_comparison(raw)
country = country_selection(country)
display(summary.round(4)); display(country.round(4))
summary.to_csv(REPORTS/'w12_missing_id.csv'); country.to_csv(REPORTS/'w12_country_selection.csv')
from src.publication import MIN_GROUP, POOLED
print(f'Nhóm quốc gia có >= {MIN_GROUP} dòng mua hợp lệ nhưng không có dòng biết ID:', ', '.join(country.index[(country.total>=MIN_GROUP)&(country.Known_ID==0)&(country.index!=POOLED)]))
clean, _ = clean_transactions(raw)
del raw
transactions = integrate_external_sources(clean)
del clean
snapshots = ce.temporal_datasets(transactions, observed_until, include_spend_target=True)
del transactions


Nguồn: online_retail_II.csv | 1,067,371 dòng


,all_lines,purchase_lines,gross_value_gbp,median_line_gbp,median_quantity,median_unit_price_gbp,all_lines_share,purchase_lines_share,gross_value_gbp_share
id_status,,,,,,,,,
Known_ID,797885,776844,1.708113e+07,12.48,6.0,1.95,0.7724,0.774,0.8689
Missing_ID,235151,226857,2.577915e+06,5.02,1.0,3.29,0.2276,0.226,0.1311


id_status,Known_ID,Missing_ID,total,missing_share,known_line_share
Country,,,,,
United Kingdom,699684,224036,923720,0.2425,0.9007
EIRE,15529,1583,17112,0.0925,0.0200
Germany,15788,0,15788,0.0000,0.0203
France,13036,128,13164,0.0097,0.0168
Netherlands,4981,0,4981,0.0000,0.0064
Spain,3560,0,3560,0.0000,0.0046
Switzerland,2951,117,3068,0.0381,0.0038
Belgium,2911,0,2911,0.0000,0.0037
Portugal,2295,114,2409,0.0473,0.0030


Nhóm quốc gia có >= 30 dòng mua hợp lệ nhưng không có dòng biết ID: Hong Kong, Bermuda
[Merge 1 - Country Metadata] Trước: 794,437 | Sau: 794,437 | Tỷ lệ khớp: 99.71%
  -> Các giá trị chưa khớp (2,273 dòng): ['Channel Islands', 'Unspecified', 'West Indies', 'European Community']
[Merge 2 - Season Calendar]  Trước: 794,437 | Sau: 794,437 | Tỷ lệ khớp: 100.00%


## 1. Độ đại diện và quyền riêng tư
Đếm dòng mua hợp lệ sau chuẩn hóa/loại duplicate và trước bỏ ID; không suy ra số khách thiếu định danh. Nội bộ UK/International/Unknown theo modal geography trước cutoff; khi công bố luôn gộp International và Unknown thành Non-UK / Unknown. Không công bố thống kê International riêng hoặc gap riêng. Nếu một nhóm nhỏ, ẩn cả hai nhóm để tránh suy ngược từ Overall. Chỉ so sánh nội bộ nhà bán lẻ, không có mẫu dân số đối chứng. k tính theo số khách phân biệt trong test; suppression k<10 chỉ minh họa, không loại khách khỏi test. Không xuất lớp tương đương hay dòng cấp khách. Quy tắc công bố giảm rủi ro suy ngược trong báo cáo mới, không xóa dữ liệu đã công bố trong lịch sử Git hoặc chứng minh ẩn danh.
Bảng quốc gia dùng cùng hàm công bố với notebook 01; ngưỡng 30 dòng mua, gộp ít nhất hai quốc gia khi có nhóm nhỏ. Notebook 02 dùng ngưỡng 30 khách distinct. Đây không phải bảo đảm ẩn danh.

In [2]:
representation = public_representation(snapshots)
display(representation); display(representation.div(representation.sum(axis=0, min_count=1)).round(4))
representation.to_csv(REPORTS/'w12_representation.csv')
privacy = privacy_summary(snapshots['test'], k_target=10)
display(privacy); privacy.to_csv(REPORTS/'w12_privacy.csv')


,train,validation,test
market,,,
UK,4149.0,4503.0,4773.0
Non-UK / Unknown,364.0,406.0,452.0


,train,validation,test
market,,,
UK,0.9193,0.9173,0.9135
Non-UK / Unknown,0.0807,0.0827,0.0865


,customers,suppressed,k_min,singleton_customers,customers_below_k,k_target
scenario,,,,,,
exact,5225,0,1,5225,5225,10
generalized,5225,0,1,11,172,10
generalized_suppressed,5053,172,10,0,0,10


## 2. Classification theo thị trường
Cố định Random Forest thắng CV; tái tạo ngưỡng F1 từ validation. Không dùng test để fit/chọn ngưỡng. So sánh recall/TPR, FPR, accuracy, precision, ROC-AUC, AP. Báo cáo public chỉ có Overall, UK và Non-UK / Unknown. Nếu một nhóm có n<30, ẩn số liệu của cả hai nhóm; nếu số lớp thực hoặc lớp dự báo của một nhóm <10, ẩn metric của cả hai nhóm. N/A không phải 0. Gap Non-UK / Unknown − UK, không phải gap International. Ghép các đầu vào bằng index Customer ID duy nhất theo đúng thứ tự; không xuất index. Không tune threshold theo nhóm.

In [3]:
X_train, y_train = ce.split_xy(snapshots['train'])
X_val, y_val = ce.split_xy(snapshots['validation'])
clf = ce.make_pipeline(ce.candidate_models()['Random Forest']).fit(X_train, y_train)
val_prob = clf.predict_proba(X_val)[:,1]
threshold, val_f1 = ce.choose_threshold(y_val, val_prob)
test = snapshots['test'].set_index('customer_id', drop=False, verify_integrity=True)
X_test, y_test = ce.split_xy(test)
prob = pd.Series(clf.predict_proba(X_test)[:,1], index=X_test.index)
y_test_keyed = pd.Series(y_test, index=X_test.index)
groups = market_groups(test)
clf_table = classification_by_group(y_test_keyed, prob, groups, threshold)
print(f'Threshold từ validation (full precision): {threshold:.12g}; F1 validation {val_f1:.4f}')
display(clf_table.round(4))
for col in ['recall','fpr','accuracy','roc_auc']:
    print(f'Non-UK / Unknown − UK {col}: {signed_gap(clf_table,col):+.4f}')
# Chỉ ghi CSV metric nhóm sau khi qua cổng xác thực.


Threshold từ validation (full precision): 0.480468678468; F1 validation 0.6602


,n,share,positive_support,negative_support,prevalence,selection_rate,accuracy,recall,fpr,precision,f1,roc_auc,average_precision,status
market,,,,,,,,,,,,,,
Overall,5225,1.0000,2236,2989,0.4279,0.3610,0.7301,0.6064,0.1773,0.7190,0.6579,0.7936,0.7639,descriptive
UK,4773,0.9135,2026,2747,0.4245,0.3537,0.7301,0.5987,0.1729,0.7186,0.6532,0.7927,0.7606,descriptive
Non-UK / Unknown,452,0.0865,210,242,0.4646,0.4381,0.7301,0.6810,0.2273,0.7222,0.7010,0.8004,0.7943,descriptive


Non-UK / Unknown − UK recall: +0.0822
Non-UK / Unknown − UK fpr: +0.0544
Non-UK / Unknown − UK accuracy: -0.0001
Non-UK / Unknown − UK roc_auc: +0.0077


## 3. Regression theo thị trường
Cố định Dummy median thắng CV; tái fit trên train. Không chọn lại Ridge/OLS dựa vào test. MAE là chỉ tiêu chính; mean error = actual − predicted (dương = dự báo thiếu), RMSE nhạy với đơn lớn, R² không xác định nếu target hằng. So sánh cả mean actual và zero share. Không loại ngoại lai. Giữ đúng phân hoạch và quy tắc ẩn cả hai nhóm nếu một nhóm n<30. MAE khác nhau có thể do phân phối chi tiêu khác, không tự chứng minh bất công. Winner dự báo 0 GBP, không phù hợp để dự báo doanh thu hoặc phân khúc.

In [4]:
X_reg_train, y_reg_train = re.split_xy(snapshots['train'])
reg = re.make_pipeline(re.candidate_models()['Dummy median']).fit(X_reg_train, y_reg_train)
X_reg_val, y_reg_val = re.split_xy(snapshots['validation'])
X_reg_test, y_reg_test = re.split_xy(test)
pred = pd.Series(re.predict_gbp(reg, X_reg_test), index=X_reg_test.index)
y_reg_keyed = pd.Series(y_reg_test, index=X_reg_test.index)
if not y_test_keyed.index.equals(y_reg_keyed.index) or not y_test_keyed.eq(y_reg_keyed.gt(0).astype(int)).all():
    raise ValueError('Classification và Regression test không khớp.')
reg_table = regression_by_group(y_reg_keyed, pred, groups)
display(reg_table.round(4))
for col in ['mae_gbp','rmse_gbp','mean_error_gbp']:
    print(f'Non-UK / Unknown − UK {col}: {signed_gap(reg_table,col):+.2f} GBP')
# Chỉ ghi CSV metric nhóm sau khi qua cổng xác thực.


,n,share,mae_gbp,rmse_gbp,r2,mean_error_gbp,mean_actual_gbp,mean_predicted_gbp,zero_share,status
market,,,,,,,,,,
Overall,5225,1.0000,515.3416,3144.9414,-0.0276,515.3416,515.3416,0.0,0.5721,descriptive
UK,4773,0.9135,467.8408,2787.4855,-0.0290,467.8408,467.8408,0.0,0.5755,descriptive
Non-UK / Unknown,452,0.0865,1016.9378,5681.8507,-0.0331,1016.9378,1016.9378,0.0,0.5354,descriptive


Non-UK / Unknown − UK mae_gbp: +549.10 GBP
Non-UK / Unknown − UK rmse_gbp: +2894.37 GBP
Non-UK / Unknown − UK mean_error_gbp: +549.10 GBP


## 4. Xác thực và cập nhật tài liệu
Kiểm tra SHA-256 dữ liệu, fingerprint mã nguồn pipeline/model/fairness và source notebook, Python 3.11 cùng phiên bản thư viện tính toán đã pin; đối chiếu số mẫu, prevalence, threshold đầy đủ, toàn bộ metric của winner trên validation/test (Classification thêm default threshold), privacy summary và quy tắc công bố. Metric tham chiếu lấy từ notebook đã nghiệm thu; dung sai theo độ chính xác hiển thị 4 chữ số. Không tự thay đổi lock để vượt kiểm tra. Nếu không khớp, chỉ có kết quả tổng hợp cục bộ, không cập nhật Model Card/Datasheet. Kiểm tra marker của cả ba tài liệu trước khi ghi. Đây là guard tái lập, không phải chữ ký chống sửa đổi hoặc chứng minh mô hình tuyệt đối chính xác.

In [5]:
classification_actual = classification_reference(ce, y_train, y_val, val_prob, y_test, prob, threshold, val_f1)
regression_actual = regression_reference(re, reg, X_reg_train, y_reg_train, X_reg_val, y_reg_val, X_reg_test, y_reg_test)
issues = reference_issues(ROOT, classification_actual, regression_actual, privacy)
issues.extend(publication_issues(representation, clf_table, reg_table))
print('Reference checks:', 'PASS' if not issues else 'FAIL')
if issues:
    print('Chưa ghi tài liệu tham chiếu: ' + '; '.join(issues))
else:
    update_cards([
        (ROOT/'docs/model_card_classification.md', clf_table,
         ['n','accuracy','recall','fpr','precision','roc_auc'],
         f'Temporal test đã xác thực; threshold validation {threshold:.12g}; gap recall Non-UK / Unknown − UK {signed_gap(clf_table,"recall"):+.4f}.'),
        (ROOT/'docs/model_card_regression.md', reg_table,
         ['n','mae_gbp','rmse_gbp','r2','mean_error_gbp'],
         f'Temporal test đã xác thực; gap MAE Non-UK / Unknown − UK {signed_gap(reg_table,"mae_gbp"):+.2f} GBP.'),
        (ROOT/'docs/datasheet.md', privacy,
         ['customers','suppressed','k_min','customers_below_k','k_target'],
         'Mô phỏng generalize/suppress trên test; không công bố bảng khách, không khẳng định dữ liệu gốc ẩn danh.')
    ])
    clf_table.to_csv(REPORTS/'w12_classification_groups.csv')
    reg_table.to_csv(REPORTS/'w12_regression_groups.csv')
    print('Ba tài liệu đã cập nhật aggregate; nhóm cần review trước khi commit.')


Reference checks: PASS
Ba tài liệu đã cập nhật aggregate; nhóm cần review trước khi commit.


## 5. Kết luận và checklist Ethics
- Nguồn và giới hạn được ghi trong datasheet. Không scrape trong pipeline.
- Không công bố mã ID hoặc bảng khách; đo k thực nghiệm, có suppression; lịch sử Git và dữ liệu nguồn vẫn có rủi ro.
- Định lượng missing ID và độ đại diện; UK chi phối kết quả chung. Không so sánh nhóm quốc tế riêng trong báo cáo công khai; gap của nhóm gộp chỉ mang tính mô tả.
- Không chọn lại model theo test. Không triển khai Dummy median; dùng Classification thận trọng trong nghiên cứu hồi cứu.
- Review và nộp model card một trang/mô hình cùng báo cáo. Không cần Spark cho quy mô này.